# Stage 7 - comprehensive attack grid (FGSM / PGD / HopSkipJump x baseline / static-AT / Madry-AT / RF)

Notebook 05 settled the decisive comparison with **PGD only** under two threat
models. This notebook widens that to a full grid so the robustness picture does
not rest on a single attack:

* **Attacks.** *FGSM* (one-step gradient), *PGD* (iterative gradient), and
  *HopSkipJump* (decision-based **black-box** - no gradients).
* **Models.** baseline CNN, **static** adversarial training (augment-once from a
  frozen baseline), **Madry** adversarial training (iterative min-max), and the
  Random Forest.
* **Threat models.** *transfer* = the attack is crafted once from the **undefended
  baseline** and fed to every model. *white-box* = the attack is crafted **against
  the model it hits** (its own gradients). White-box applies to the gradient
  models only; the Random Forest has no gradients, so FGSM/PGD reach it in the
  transfer column only.
* **Black-box (HopSkipJump)** has no gradient, so there is no "white-box" column
  for it: it is crafted **directly against each deployed model** using only its
  decisions - and because it needs only labels, the Random Forest gets a cell too.

Metric throughout: **robust-support macro-F1** (the classes with >=2 held-out test
frames), the same headline metric as notebook 05, at **PGD/FGSM epsilon = 0.10**.

This notebook is **additive**: it writes `results/<name>_attack_grid_results.json`
and never modifies the existing defence / adversarial result files.


In [1]:
import sys
from pathlib import Path
try:
    import adversec
except ModuleNotFoundError:
    sys.path.insert(0, str(Path.cwd().parent)); import adversec
import numpy as np, pandas as pd, io, contextlib, json
from adversec import config
from adversec.contract import FEATURES, LABEL_COLUMN, ID_COLUMN, DATA_COLUMNS
import torch, joblib
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight
from adversec.models import CNN1D, train_cnn, build_random_forest
from adversec.experiments import attack as atk, realism
from adversec.experiments.defense import adversarial_train_cnn, madry_adversarial_train_cnn
from art.estimators.classification import SklearnClassifier
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device:', DEVICE)

DATASETS = ['ciciov2024', 'road']
EPS               = 0.10   # decisive epsilon, matching notebooks 05 and 06
N_REPEATS         = 5      # gradient-attack grid repeats (paired seeds, like nb05)
HSJ_REPEATS       = 2      # HopSkipJump is the cost driver -> fewer repeats (subset of N_REPEATS)
HSJ_MAX_PER_CLASS = 20     # stratified subsample per class (matches nb06)
HSJ_KWARGS = dict(max_iter=15, max_eval=300, init_eval=30, init_size=30)  # reduced budget -> LOWER BOUND (see attack.py / nb06)

id_idx = FEATURES.index(ID_COLUMN); data_idx = [FEATURES.index(c) for c in DATA_COLUMNS]
def mf1(y, p, labels=None): return f1_score(y, p, labels=labels, average='macro', zero_division=0)

# Load arrays once. robust-support = classes with >=2 test frames (same rule as nb05).
# Also learn the per-ID benign envelope (TRAIN benign only) so the black-box cells can
# report what share of crafted frames would actually be injectable.
data = {}
for name in DATASETS:
    a = np.load(config.PROCESSED_DIR / f'{name}_stage2_arrays.npz')
    classes = list(joblib.load(config.PROCESSED_DIR / f'{name}_label_encoder.joblib').classes_)
    cfg = config.load_dataset_config(name)
    scaler = joblib.load(config.PROCESSED_DIR / f'{name}_feature_scaler.joblib')
    counts = pd.Series(a['y_test']).value_counts()
    rs = [i for i in range(len(classes)) if counts.get(i, 0) >= 2]
    train_dup = pd.read_csv(config.PROCESSED_DIR / f'{name}_train_dup.csv')
    benign = cfg['benign_label']
    ranges = realism.learn_observed_ranges(train_dup[train_dup[LABEL_COLUMN] == benign], ID_COLUMN, DATA_COLUMNS)
    data[name] = dict(Xtr=a['X_train'], ytr=a['y_train'], Xte=a['X_test'].astype(np.float32),
                      yte=a['y_test'], classes=classes, cfg=cfg, rs=rs, scaler=scaler, ranges=ranges)
    print(f'{name}: {len(classes)} classes; robust-support = {[classes[i] for i in rs]}')

device: cuda
ciciov2024: 6 classes; robust-support = ['DoS', 'benign', 'spoofing-RPM']
road: 5 classes; robust-support = ['benign', 'fuzzing', 'max-speedometer', 'reverse-light-off', 'reverse-light-on']


## The engine: one full grid from one seed

`one_run_grid` trains all four models once for a seed, then measures every cell.
The three CNNs are trained inside a redirected-stdout block by default so the
averaging loop stays quiet; the transfer attack is crafted from the standalone
baseline (exactly as in nb05), and each white-box attack is crafted from the model
it targets. `do_hsj=True` additionally runs the black-box HopSkipJump section on
the same, already-trained models (no retraining).


In [2]:
def _class_weights(cfg, ytr):
    if not cfg.get('cnn_class_weights'):
        return None
    w = compute_class_weight('balanced', classes=np.unique(ytr), y=ytr)
    return torch.tensor(w, dtype=torch.float32, device=DEVICE)


def _hsj_direct(name, targets, seed):
    """Black-box HopSkipJump crafted DIRECTLY against each model (decisions only,
    no gradients) on a stratified subsample. Reports robust-support macro-F1, the
    mean L2 perturbation, and the per-ID envelope rejection % -- so the black-box
    drop is read next to how many of its frames a benign-range filter would reject
    as non-injectable. `targets` maps model label -> an ART classifier (CNNs already
    wrapped; the RF wrapped in SklearnClassifier, since HSJ needs only labels)."""
    d = data[name]; Xte, yte, rs, scaler, ranges = d['Xte'], d['yte'], d['rs'], d['scaler'], d['ranges']
    rng = np.random.RandomState(seed)
    idx_by_class = {c: np.where(yte == c)[0] for c in np.unique(yte)}
    sub = np.concatenate([rng.choice(ix, size=min(HSJ_MAX_PER_CLASS, len(ix)), replace=False)
                          for ix in idx_by_class.values()])
    Xs, ys = Xte[sub], yte[sub]
    res = {'_meta': dict(n_samples=int(len(Xs)), max_per_class=HSJ_MAX_PER_CLASS, budget=dict(HSJ_KWARGS)),
           '_clean_subsample_f1': {}}
    for label, clf in targets.items():
        res['_clean_subsample_f1'][label] = float(mf1(ys, clf.predict(Xs).argmax(1), rs))
        Xh = atk.generate_hopskipjump(clf, Xs, **HSJ_KWARGS)
        f1 = mf1(ys, clf.predict(Xh).argmax(1), rs)
        _, Xh_int = realism.round_to_integer_frames(Xh, scaler)
        rej = 100.0 * (~realism.observed_range_mask(Xh_int, ranges, id_idx, data_idx)).sum() / len(Xh)
        l2 = float(np.linalg.norm((Xh - Xs).reshape(len(Xs), -1), axis=1).mean())
        res[label] = dict(f1=float(f1), pct_rejected_by_envelope=round(float(rej), 1), mean_l2=round(l2, 4))
    return res


def one_run_grid(name, seed=42, do_hsj=False, quiet_train=True):
    d = data[name]
    Xtr, ytr, Xte, yte, classes, cfg, rs = d['Xtr'], d['ytr'], d['Xte'], d['yte'], d['classes'], d['cfg'], d['rs']
    cw = _class_weights(cfg, ytr)
    ctx = contextlib.redirect_stdout(io.StringIO()) if quiet_train else contextlib.nullcontext()
    with ctx:
        base   = train_cnn(CNN1D(Xtr.shape[1], len(classes)), Xtr, ytr, n_epochs=50, device=DEVICE, class_weights=cw, random_seed=seed)
        static = adversarial_train_cnn(Xtr, ytr, strategy='pgd', n_epochs=50, device=DEVICE, class_weights=cw, random_seed=seed)
        madry  = madry_adversarial_train_cnn(Xtr, ytr, n_epochs=50, device=DEVICE, class_weights=cw, random_seed=seed)
    rf = build_random_forest(random_seed=seed).fit(Xtr, ytr)
    bcf = atk.wrap_cnn_for_art(base,   Xtr.shape[1], len(classes), DEVICE)
    scf = atk.wrap_cnn_for_art(static, Xtr.shape[1], len(classes), DEVICE)
    mcf = atk.wrap_cnn_for_art(madry,  Xtr.shape[1], len(classes), DEVICE)

    def f(clf, X):  return mf1(yte, clf.predict(X).argmax(1), rs)
    def frf(X):     return mf1(yte, rf.predict(X), rs)

    # transfer attacks: crafted ONCE from the undefended baseline, fed to everyone.
    fgsm_t = atk.generate_fgsm(bcf, Xte, EPS)
    pgd_t  = atk.generate_pgd(bcf, Xte, EPS, verbose=False)
    # white-box attacks: crafted against each defended model itself (gradient models only).
    fgsm_s = atk.generate_fgsm(scf, Xte, EPS); fgsm_m = atk.generate_fgsm(mcf, Xte, EPS)
    pgd_s  = atk.generate_pgd(scf, Xte, EPS, verbose=False); pgd_m = atk.generate_pgd(mcf, Xte, EPS, verbose=False)

    out = {
        'clean':         {'baseline': f(bcf, Xte),  'static': f(scf, Xte),  'madry': f(mcf, Xte),  'rf': frf(Xte)},
        'fgsm_transfer': {'baseline': f(bcf, fgsm_t), 'static': f(scf, fgsm_t), 'madry': f(mcf, fgsm_t), 'rf': frf(fgsm_t)},
        'fgsm_whitebox': {'static': f(scf, fgsm_s), 'madry': f(mcf, fgsm_m)},
        'pgd_transfer':  {'baseline': f(bcf, pgd_t),  'static': f(scf, pgd_t),  'madry': f(mcf, pgd_t),  'rf': frf(pgd_t)},
        'pgd_whitebox':  {'static': f(scf, pgd_s), 'madry': f(mcf, pgd_m)},
    }
    # For the undefended baseline, the transfer attack IS its own white-box attack
    # (it is crafted from the baseline), so mirror it into the white-box row.
    out['fgsm_whitebox']['baseline'] = out['fgsm_transfer']['baseline']
    out['pgd_whitebox']['baseline']  = out['pgd_transfer']['baseline']

    if do_hsj:
        targets = {'baseline': bcf, 'static': scf, 'madry': mcf,
                   'rf': SklearnClassifier(model=rf, clip_values=(0.0, 1.0))}
        out['hopskipjump'] = _hsj_direct(name, targets, seed)
    return out

## One illustrative run (see a single full grid)

A single seed is not trustworthy on CICIoV2024 (CUDA nondeterminism on tiny data);
the averaged tables below are the ones to quote. This is just to see the shape.


In [3]:
CELLS  = ['clean', 'fgsm_transfer', 'fgsm_whitebox', 'pgd_transfer', 'pgd_whitebox']
MODELS = ['baseline', 'static', 'madry', 'rf']
ROW_LABEL = {'clean': 'clean', 'fgsm_transfer': 'FGSM transfer', 'fgsm_whitebox': 'FGSM white-box',
             'pgd_transfer': 'PGD transfer', 'pgd_whitebox': 'PGD white-box'}

def _cell_str(res, cellkey, m, fmt):
    if m in res[cellkey]:
        return fmt(res[cellkey][m])
    return 'n/a'

for name in DATASETS:
    print('########## ' + name + ' ##########')
    r = one_run_grid(name, seed=42, do_hsj=True, quiet_train=True)
    print(f'{name}: robust-support macro-F1, eps={EPS}')
    print(f"  {'attack / threat':18s}{'baseline':>11}{'static-AT':>11}{'Madry-AT':>11}{'RandForest':>12}")
    for ck in CELLS:
        vals = ''.join(f'{_cell_str(r, ck, m, lambda x: f"{x:.3f}"):>{11 if m!="rf" else 12}}' for m in MODELS)
        print(f"  {ROW_LABEL[ck]:18s}{vals}")
    h = r['hopskipjump']
    print(f"  {'HSJ black-box':18s}" + ''.join(f'{h[m]["f1"]:>{11 if m!="rf" else 12}.3f}' for m in MODELS)
          + f"   (subsample={h['_meta']['n_samples']}, LOWER BOUND)")

########## ciciov2024 ##########


PGD - Batches:   0%|          | 0/120 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/120 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/120 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/120 [00:00<?, ?it/s]

...

PGD - Batches:   0%|          | 0/997 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/997 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/997 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/997 [00:00<?, ?it/s]

road: robust-support macro-F1, eps=0.1
  attack / threat      baseline  static-AT   Madry-AT  RandForest
  clean                   0.997      0.997      0.767       1.000
  FGSM transfer           0.392      0.596      0.532       0.139
  FGSM white-box          0.392      0.299      0.418         n/a
  PGD transfer            0.326      0.687      0.508       0.139
  PGD white-box           0.326      0.271      0.482         n/a
  HSJ black-box           0.064      0.073      0.064       0.067   (subsample=100, LOWER BOUND)


## The averaged grid (mean +/- std over repeats)

The full grid, averaged over `N_REPEATS` paired seeds. HopSkipJump runs on the
first `HSJ_REPEATS` of those seeds (it is the expensive part) and reuses the
models already trained for that seed.


In [4]:
agg = {name: {ck: {m: [] for m in MODELS} for ck in CELLS} for name in DATASETS}
hsj_agg = {name: {m: {'f1': [], 'pct_rejected_by_envelope': [], 'mean_l2': []} for m in MODELS} for name in DATASETS}

for r in range(N_REPEATS):
    for name in DATASETS:
        res = one_run_grid(name, seed=42 + r, do_hsj=(r < HSJ_REPEATS), quiet_train=True)
        for ck in CELLS:
            for m in MODELS:
                if m in res[ck]:
                    agg[name][ck][m].append(res[ck][m])
        if 'hopskipjump' in res:
            for m in MODELS:
                for k in ('f1', 'pct_rejected_by_envelope', 'mean_l2'):
                    hsj_agg[name][m][k].append(res['hopskipjump'][m][k])
    print(f'repeat {r + 1}/{N_REPEATS} done' + (' (+HSJ)' if r < HSJ_REPEATS else ''))

def msr(v):
    return f'{np.mean(v):.3f}+/-{np.std(v):.3f}' if len(v) else 'n/a'

print()
for name in DATASETS:
    print(f'===== {name}: robust-support macro-F1, mean +/- std over {N_REPEATS} repeats, eps={EPS} =====')
    print(f"  {'attack / threat':18s}{'baseline':>16}{'static-AT':>16}{'Madry-AT':>16}{'RandForest':>16}")
    for ck in CELLS:
        print(f"  {ROW_LABEL[ck]:18s}" + ''.join(f'{msr(agg[name][ck][m]):>16}' for m in MODELS))
    print(f"  {'HSJ black-box':18s}" + ''.join(f'{msr(hsj_agg[name][m]["f1"]):>16}' for m in MODELS)
          + f'   (black-box, LOWER BOUND, {HSJ_REPEATS} repeats)')
    print()

PGD - Batches:   0%|          | 0/120 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/120 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/120 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/120 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/120 [00:00<?, ?it/s]

...

PGD - Batches:   0%|          | 0/997 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/997 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/997 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/997 [00:00<?, ?it/s]

repeat 5/5 done

===== ciciov2024: robust-support macro-F1, mean +/- std over 5 repeats, eps=0.1 =====
  attack / threat           baseline       static-AT        Madry-AT      RandForest
  clean                0.830+/-0.055   0.859+/-0.015   0.797+/-0.053   0.859+/-0.022
  FGSM transfer        0.305+/-0.075   0.694+/-0.127   0.560+/-0.148   0.312+/-0.008
  FGSM white-box       0.305+/-0.075   0.294+/-0.016   0.465+/-0.107             n/a
  PGD transfer         0.304+/-0.075   0.759+/-0.041   0.532+/-0.101   0.345+/-0.088
  PGD white-box        0.304+/-0.075   0.277+/-0.007   0.393+/-0.069             n/a
  HSJ black-box        0.100+/-0.038   0.038+/-0.038   0.280+/-0.023   0.369+/-0.057   (black-box, LOWER BOUND, 2 repeats)

===== road: robust-support macro-F1, mean +/- std over 5 repeats, eps=0.1 =====
  attack / threat           baseline       static-AT        Madry-AT      RandForest
  clean                0.997+/-0.001   0.996+/-0.002   0.796+/-0.060   1.000+/-0.000
  FGSM transf

## How injectable are the black-box frames?

HopSkipJump minimises L2 but ignores CAN legality, so its frames are not
necessarily injectable. This is the same per-ID benign-envelope check as nb06,
now reported per target model: a high rejection % means most of that black-box
drop would be filtered before it ever reached the bus.


In [5]:
for name in DATASETS:
    print(f'===== {name}: HopSkipJump (black-box, direct) -- feasibility of the crafted frames =====')
    print(f"  {'model':12s}{'HSJ F1':>14}{'mean L2':>14}{'% rejected by envelope':>26}")
    for m in MODELS:
        h = hsj_agg[name][m]
        print(f"  {m:12s}{msr(h['f1']):>14}{msr(h['mean_l2']):>14}{msr(h['pct_rejected_by_envelope']):>26}")
    print()

===== ciciov2024: HopSkipJump (black-box, direct) -- feasibility of the crafted frames =====
  model               HSJ F1       mean L2    % rejected by envelope
  baseline     0.100+/-0.038 0.324+/-0.029           89.650+/-10.350
  static       0.038+/-0.038 0.232+/-0.007            94.850+/-5.150
  madry        0.280+/-0.023 0.071+/-0.023            43.100+/-8.600
  rf           0.369+/-0.057 0.142+/-0.075           51.750+/-10.350

===== road: HopSkipJump (black-box, direct) -- feasibility of the crafted frames =====
  model               HSJ F1       mean L2    % rejected by envelope
  baseline     0.068+/-0.002 0.112+/-0.003            79.500+/-1.500
  static       0.056+/-0.001 0.103+/-0.003            86.000+/-1.000
  madry        0.136+/-0.072 0.176+/-0.006            80.500+/-1.500
  rf           0.067+/-0.000 0.003+/-0.000            66.000+/-1.000



## Save the grid (new file: `results/<name>_attack_grid_results.json`)

In [6]:
def ms_full(v):
    arr = np.array(v, dtype=float)
    return {'mean': float(arr.mean()), 'std': float(arr.std()), 'runs': [float(x) for x in arr]} if len(arr) else None

config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
for name in DATASETS:
    d = data[name]
    grid = {}
    for ck in CELLS:
        grid[ck] = {m: ms_full(agg[name][ck][m]) for m in MODELS if len(agg[name][ck][m])}
    hsj = {'n_repeats': HSJ_REPEATS,
           'max_per_class': HSJ_MAX_PER_CLASS,
           'budget': dict(HSJ_KWARGS),
           'note': ('crafted directly against each model (decisions only). Reduced query budget -> '
                    'F1 is a LOWER BOUND on a well-resourced black-box attacker. pct_rejected_by_envelope '
                    '= share of crafted frames the per-ID benign envelope would reject as non-injectable.'),
           'per_model': {m: {'f1': ms_full(hsj_agg[name][m]['f1']),
                             'pct_rejected_by_envelope': ms_full(hsj_agg[name][m]['pct_rejected_by_envelope']),
                             'mean_l2': ms_full(hsj_agg[name][m]['mean_l2'])} for m in MODELS}}
    report = {
        'dataset': name,
        'method': 'comprehensive_attack_grid',
        'description': (
            'Robust-support macro-F1 across FGSM / PGD / HopSkipJump x baseline / static-AT / '
            'Madry-AT / Random Forest, under the transfer and white-box threat models, at eps='
            f'{EPS}, averaged over {N_REPEATS} paired-seed repeats. transfer = attack crafted from '
            'the undefended baseline CNN and fed to every model. white_box = attack crafted against '
            'the model it hits (gradient models only; the Random Forest has no gradients). '
            'HopSkipJump is black-box (decisions only) and crafted directly against each model.'),
        'attack_eps': EPS,
        'n_repeats': N_REPEATS,
        'metric': 'robust-support macro-F1 (classes with >=2 held-out test frames)',
        'robust_support_classes': [d['classes'][i] for i in d['rs']],
        'models': ['baseline_cnn', 'static_at_cnn', 'madry_at_cnn', 'random_forest'],
        'grid': grid,
        'blackbox_hopskipjump_direct': hsj,
        'note': ('Additive artifact -- separate from ' + name + '_defence_results.json, which it does '
                 'not modify. Same prepared inputs as notebooks 05/06.'),
    }
    path = config.RESULTS_DIR / f'{name}_attack_grid_results.json'
    path.write_text(json.dumps(report, indent=2))
    print('saved ->', path)

saved -> /home/koala/lab/adversec/results/ciciov2024_attack_grid_results.json
saved -> /home/koala/lab/adversec/results/road_attack_grid_results.json


## Does the attack *type* matter, or do the columns just move together?

The question this grid exists to answer: are FGSM, PGD and HopSkipJump
interchangeable (their columns rise and fall together), or does the ranking of the
defences depend on which attack you pick? Two paired t-tests per dataset make the
key contrasts explicit, on the paired-seed runs (no retraining):

* **FGSM vs PGD white-box on the Madry model** -- is the iterative gradient attack
  meaningfully stronger than the one-step one against the strongest defence?
* **Madry vs static, white-box** -- does nb05's PGD-only finding (that the
  iterative form survives white-box where the static form collapses) still hold
  under FGSM?


In [7]:
from scipy import stats as sstats

def paired(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    if len(a) < 2 or len(a) != len(b):
        return None
    t, p = sstats.ttest_rel(a, b)
    return {'mean_diff': float(a.mean() - b.mean()), 't_statistic': float(t),
            'p_value': float(p), 'significant_at_0.05': bool(p < 0.05)}

sig = {}
print(f"  {'dataset':12s}{'comparison':38s}{'n':>4}{'mean diff':>12}{'t':>8}{'p':>10}   sig?")
for name in DATASETS:
    A = agg[name]
    tests = {
        'madry_whitebox_pgd_vs_fgsm': paired(A['pgd_whitebox']['madry'], A['fgsm_whitebox']['madry']),
        'whitebox_madry_vs_static_fgsm': paired(A['fgsm_whitebox']['madry'], A['fgsm_whitebox']['static']),
        'whitebox_madry_vs_static_pgd': paired(A['pgd_whitebox']['madry'], A['pgd_whitebox']['static']),
    }
    sig[name] = tests
    labels = {'madry_whitebox_pgd_vs_fgsm': 'Madry WB: PGD vs FGSM',
              'whitebox_madry_vs_static_fgsm': 'WB FGSM: Madry vs static',
              'whitebox_madry_vs_static_pgd': 'WB PGD: Madry vs static'}
    for key, v in tests.items():
        if v is None:
            print(f"  {name:12s}{labels[key]:38s}{'n/a (need >=2 repeats)':>40}")
        else:
            print(f"  {name:12s}{labels[key]:38s}{N_REPEATS:>4}{v['mean_diff']:>+12.3f}"
                  f"{v['t_statistic']:>8.2f}{v['p_value']:>10.4f}   {'YES' if v['significant_at_0.05'] else 'no'}")

# merge into the saved grid file
for name in DATASETS:
    path = config.RESULTS_DIR / f'{name}_attack_grid_results.json'
    existing = json.loads(path.read_text())
    existing['paired_significance'] = sig[name]
    path.write_text(json.dumps(existing, indent=2))
    print('updated ->', path)

  dataset     comparison                               n   mean diff       t         p   sig?
  ciciov2024  Madry WB: PGD vs FGSM                    5      -0.072   -1.61    0.1818   no
  ciciov2024  WB FGSM: Madry vs static                 5      +0.172    2.85    0.0464   YES
  ciciov2024  WB PGD: Madry vs static                  5      +0.116    3.71    0.0207   YES
  road        Madry WB: PGD vs FGSM                    5      -0.027   -1.01    0.3699   no
  road        WB FGSM: Madry vs static                 5      +0.337    6.72    0.0026   YES
  road        WB PGD: Madry vs static                  5      +0.381    9.29    0.0007   YES
updated -> /home/koala/lab/adversec/results/ciciov2024_attack_grid_results.json
updated -> /home/koala/lab/adversec/results/road_attack_grid_results.json
